# 06 - SHAP analysis

- **Figs. 3 and 4.** Top-10 SHAP importance for each of the eight feature cases, for ASP and BAU fields.
  The nitrogen zone is left out for BAU, where a single uniform rate was applied.
- **Supplementary Figs. S6-S11.** The same plots for dry, normal and wet years.
- **Figs. 5 and 6.** Maps of feature values and their SHAP values within one field-year.

The models here are fitted on all pixels of a group, because the aim is explanation, not scoring.

In [ ]:
import sys
from pathlib import Path

import numpy as np
from xgboost import XGBRegressor

sys.path.append("../src")
from yieldml import CASES, CASE_LABELS, TARGET, Y_MULT, by_year_class, load_all

import matplotlib.pyplot as plt
import shap
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

In [ ]:
FIG_DIR = Path("../results/figures")
data = load_all()
TOP_N = 10
SHORT = {"Precipitation": "P"}

## Plot helpers

In [ ]:
def fit_and_explain(df, feats, n_estimators):
    feats = [f for f in feats if f in df.columns]
    X = df[feats]
    model = XGBRegressor(n_estimators=n_estimators, max_depth=3, learning_rate=0.01, random_state=42)
    model.fit(X, df[TARGET] * Y_MULT)
    return model, shap.Explainer(model)(X)


def bar_and_beeswarm(ax_bar, ax_bee, sv, top_n=TOP_N):
    mean_abs = np.abs(sv.values).mean(axis=0)
    idx = np.argsort(mean_abs)[::-1][:top_n]
    names = [SHORT.get(f, f) for f in np.array(sv.feature_names)[idx]]

    bars = ax_bar.barh(names, mean_abs[idx])
    ax_bar.invert_yaxis()
    ax_bar.set_xlabel("mean(|SHAP value|)", fontsize=9)
    ax_bar.tick_params(axis="y", labelsize=8, left=False)
    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)
    for b, v in zip(bars, mean_abs[idx]):
        ax_bar.text(v, b.get_y() + b.get_height() / 2, f" {v:.1f}", va="center", ha="left", fontsize=9)

    shap.plots.beeswarm(sv[:, idx], max_display=top_n, show=False, plot_size=None, ax=ax_bee)
    ax_bee.set_xlabel("SHAP value (impact on model output)", fontsize=9)
    ax_bee.set_ylabel("")
    ax_bee.set_yticklabels([])
    ax_bee.tick_params(axis="y", left=False, labelleft=False)


def case_grid(df, n_estimators, out_name):
    fig = plt.figure(figsize=(15, 15))
    gs = GridSpec(4, 2, figure=fig, wspace=0.30, hspace=0.30)
    for i, (case, feats) in enumerate(CASES.items()):
        sub = GridSpecFromSubplotSpec(1, 2, subplot_spec=gs[divmod(i, 2)], width_ratios=[1, 2], wspace=0.30)
        ax_bar, ax_bee = fig.add_subplot(sub[0, 0]), fig.add_subplot(sub[0, 1])
        _, sv = fit_and_explain(df, feats, n_estimators)
        bar_and_beeswarm(ax_bar, ax_bee, sv)
        ax_bar.set_title(CASE_LABELS[case], fontweight="bold")
    plt.savefig(FIG_DIR / out_name, dpi=300, bbox_inches="tight")
    plt.show()

## Figs. 3 (ASP) and 4 (BAU)

In [ ]:
case_grid(data["ASP"], 800, "fig3_shap_asp.png")
case_grid(data["BAU"].drop(columns=["Nitrogen"]), 800, "fig4_shap_bau.png")

## By climate class (Supplementary Figs. S6-S11)

In [ ]:
for group in ["ASP", "BAU"]:
    for cls in ["Dry", "Normal", "Wet"]:
        print(group, cls)
        case_grid(by_year_class(data[group], cls), 500, f"supp_shap_{group.lower()}_{cls.lower()}.png")

## SHAP maps within a field (Figs. 5 and 6)

For one field-year, each selected feature is mapped next to its SHAP value (kg/ha). Positive SHAP means
the feature pushes predicted yield above the average. The model is the Case 7 model of the field group.

In [ ]:
def to_grid(values, df, pix=5.0):
    xs, ys = np.unique(df["longitude"]), np.unique(df["latitude"])
    xi, yi = {v: i for i, v in enumerate(xs)}, {v: i for i, v in enumerate(ys)}
    arr = np.full((len(ys), len(xs)), np.nan)
    for x, y, v in zip(df["longitude"], df["latitude"], values):
        arr[yi[y], xi[x]] = v
    return arr, [xs.min() - pix / 2, xs.max() + pix / 2, ys.min() - pix / 2, ys.max() + pix / 2]


def shap_maps(group_df, field, year, show, out_name, case="Case7"):
    feats = [f for f in CASES[case] if f in group_df.columns]
    _, sv = fit_and_explain(group_df, feats, 800)
    rows = np.where((group_df["FieldName"] == field) & (group_df["Year"] == str(year)))[0]
    if len(rows) == 0:
        raise ValueError(f"no pixels for {field} in {year}")
    sub = group_df.iloc[rows]

    fig, axes = plt.subplots(len(show), 2, figsize=(7, 2.6 * len(show)))
    for r, f in enumerate(show):
        j = feats.index(f)
        for c, (vals, cmap, label) in enumerate([(sub[f].values, "viridis", f),
                                                (sv.values[rows, j], "RdBu", f"SHAP of {f}")]):
            arr, extent = to_grid(vals, sub)
            kw = {}
            if c == 1:
                lim = np.nanmax(np.abs(arr))
                kw = dict(vmin=-lim, vmax=lim)
            im = axes[r, c].imshow(arr, extent=extent, origin="lower", cmap=cmap, **kw)
            fig.colorbar(im, ax=axes[r, c], shrink=0.8)
            axes[r, c].set_title(label, fontsize=9)
            axes[r, c].set_xticks([])
            axes[r, c].set_yticks([])
    fig.suptitle(f"{field} {year}", fontsize=12)
    plt.tight_layout()
    plt.savefig(FIG_DIR / out_name, dpi=300, bbox_inches="tight")
    plt.show()


shap_maps(data["ASP"], "SCD2", 2020, ["Nitrogen", "Elevation", "ETa", "SM 30 cm", "Silt (0-15 cm)", "Carbon (0-15 cm)"],
          "fig5_shap_maps_scd2_2020.png")
shap_maps(data["BAU"].drop(columns=["Nitrogen"]), "SB3", 2021,
          ["Elevation", "ETa", "Slope", "OM (0-15 cm)", "Silt (0-15 cm)", "Clay (0-15 cm)"],
          "fig6_shap_maps_sb3_2021.png")